In [4]:
import pandas as pd
import numpy as np
import os

conditions = ["CD", "UC", "Healthy"]

print("Checking FastSpar input files\n")

for cond in conditions:
    path = f"../data/processed/{cond}_network_counts_FIXED.tsv"

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"MISSING: {path}\n"
            f"Re-run NB01 Cell 6."
        )

    df = pd.read_csv(path, sep='\t', index_col=0)

    has_nan      = df.isna().sum().sum() > 0
    not_integer  = not np.allclose(df.values, df.values.astype(int))
    has_negative = (df.values < 0).any()
    has_prefix   = df.columns.str.contains('s__').any()
    sparsity     = (df == 0).sum().sum() / df.size

    errors = []
    if has_nan:      errors.append("NaN values")
    if not_integer:  errors.append("Non-integer values")
    if has_negative: errors.append("Negative values")
    if has_prefix:   errors.append("s__ prefix in column names")

    status = "READY" if not errors else "FIX NEEDED"

    print(f"{cond}_network_counts.tsv — {status}")
    print(f"  Shape:    {df.shape[0]} samples × {df.shape[1]} species")
    print(f"  Sparsity: {sparsity:.1%}")
    print(f"  Range:    {int(df.values.min())} → {int(df.values.max())}")
    print(f"  Example:  {df.columns[:3].tolist()}")
    if errors:
        for e in errors: print(f" {e}")
    print()


Checking FastSpar input files

CD_network_counts.tsv — READY
  Shape:    748 samples × 87 species
  Sparsity: 52.6%
  Range:    0 → 99948662
  Example:  ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii']

UC_network_counts.tsv — READY
  Shape:    456 samples × 87 species
  Sparsity: 50.4%
  Range:    0 → 100000000
  Example:  ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii']

Healthy_network_counts.tsv — READY
  Shape:    428 samples × 87 species
  Sparsity: 44.2%
  Range:    0 → 85290446
  Example:  ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii']



In [2]:
import pandas as pd
import numpy as np

conditions = ["CD", "UC", "Healthy"]

for cond in conditions:

    input_path = f"../data/processed/{cond}_network_counts.tsv"

    df = pd.read_csv(
        input_path,
        sep="\t",
        index_col=0
    )

    df_fixed = (df * 10000).round().astype(int)

    df_fixed[df_fixed < 0] = 0

    df_fixed = df_fixed.loc[
        :, (df_fixed.sum(axis=0) > 0)
    ]

    output_path = (
        f"../data/processed/"
        f"{cond}_network_counts_FIXED.tsv"
    )

    df_fixed.to_csv(
        output_path,
        sep="\t"
    )

    print(f"{cond} fixed:")
    print(df_fixed.shape)
    print(
        f"saved {output_path}\n"
    )

CD fixed:
(748, 87)
saved ../data/processed/CD_network_counts_FIXED.tsv

UC fixed:
(456, 87)
saved ../data/processed/UC_network_counts_FIXED.tsv

Healthy fixed:
(428, 87)
saved ../data/processed/Healthy_network_counts_FIXED.tsv



In [5]:
  for cond in ["CD", "UC", "healthy"]:
    path = f"../data/processed/{cond}_network_counts_FIXED.tsv"
    df = pd.read_csv(path, sep='\t', index_col=0)

    cols = df.columns

    print(f"\n{cond}")

    has_species_prefix = cols.str.contains("s__").sum()
    has_genus_prefix   = cols.str.contains("g__").sum()
    has_family_prefix  = cols.str.contains("f__").sum()

    print(f"s__ (species prefix): {has_species_prefix}")
    print(f"g__ (genus prefix):   {has_genus_prefix}")
    print(f"f__ (family prefix):  {has_family_prefix}")

    taxonomy_like = cols.str.contains(";").sum()
    print(f"Full taxonomy strings: {taxonomy_like}")

    print("Example columns:", cols[:5].tolist())


CD
s__ (species prefix): 0
g__ (genus prefix):   0
f__ (family prefix):  0
Full taxonomy strings: 0
Example columns: ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii', 'Alistipes_putredinis', 'Alistipes_shahii']

UC
s__ (species prefix): 0
g__ (genus prefix):   0
f__ (family prefix):  0
Full taxonomy strings: 0
Example columns: ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii', 'Alistipes_putredinis', 'Alistipes_shahii']

healthy
s__ (species prefix): 0
g__ (genus prefix):   0
f__ (family prefix):  0
Full taxonomy strings: 0
Example columns: ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii', 'Alistipes_putredinis', 'Alistipes_shahii']


In [6]:
import numpy as np
import pandas as pd

print("FINAL CHECK - Files ready for FastSpar Colab:")

for cond in ["CD", "UC", "healthy"]:
    path = f"../data/processed/{cond}_network_counts_FIXED.tsv"
    df = pd.read_csv(path, sep="\t", index_col=0)

    has_prefix = df.columns.str.contains('s__').any()

    is_integer = np.allclose(df.values, df.values.astype(int))

    non_negative = (df.values >= 0).all()

    status = "READY" if (not has_prefix and is_integer and non_negative) else "FIX NEEDED"

    print(f"\n{cond}: {status}")
    print(f"  Shape: {df.shape}")
    print(f"  Has s__ prefixes: {has_prefix} (should be False)")
    print(f"  Integer values: {is_integer} (should be True)")
    print(f"  Non-negative: {non_negative} (should be True)")
    print(f"  Example columns: {df.columns[:3].tolist()}")



FINAL CHECK - Files ready for FastSpar Colab:

CD: READY
  Shape: (748, 87)
  Has s__ prefixes: False (should be False)
  Integer values: True (should be True)
  Non-negative: True (should be True)
  Example columns: ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii']

UC: READY
  Shape: (456, 87)
  Has s__ prefixes: False (should be False)
  Integer values: True (should be True)
  Non-negative: True (should be True)
  Example columns: ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii']

healthy: READY
  Shape: (428, 87)
  Has s__ prefixes: False (should be False)
  Integer values: True (should be True)
  Non-negative: True (should be True)
  Example columns: ['Akkermansia_muciniphila', 'Alistipes_finegoldii', 'Alistipes_onderdonkii']
